# Fig. 4(a)

In [1]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scienceplots

from tqdm import tqdm
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm


# ============================
# Plot style
# ============================
plt.style.use(['science', 'nature', 'no-latex'])

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans"],  # Ubuntu 推荐
    "font.size": 20,
    "axes.labelsize": 24,
    "axes.titlesize": 30,
    "legend.fontsize": 20,
    "xtick.labelsize": 24,
    "ytick.labelsize": 24,
    "axes.linewidth": 1.5,
})


# ============================
# Directories
# ============================
responses_dir = "../results/PRS/looming/202606/"
summary_dir = "../results/Figure4/"
os.makedirs(summary_dir, exist_ok=True)


# ============================
# Metadata files
# ============================
cell_type_file = "../data/cell_type.txt"
classification_file = "../data/classification.txt"


# ============================
# Grid size
# ============================
GRID_Y = 41
GRID_X = 82


# ============================
# Parameters
# ============================
TOP_K = 25
TOP_N = 100

HIGHLIGHT_TYPES = {"DNp01", "DNp02", "DNp03", "DNp04", "DNp11"}


# ============================
# Load cell type metadata
# ============================
cell_type_df = pd.read_csv(
    cell_type_file,
    header=None,
    names=["root_id", "primary_type", "additional_type"]
)
cell_type_df["root_id"] = cell_type_df["root_id"].astype(str)
cell_type_df.set_index("root_id", inplace=True)

classification_df = pd.read_csv(
    classification_file,
    dtype={"root_id": str}
)
classification_df["root_id"] = classification_df["root_id"].astype(str)
classification_df.set_index("root_id", inplace=True)


# ============================
# Load response files
# ============================
all_files = glob.glob(os.path.join(responses_dir, "*.npz"))
responses_by_ms = {}

print("Loading response files...")

for f in tqdm(all_files):
    base = os.path.basename(f)

    try:
        x = int(base.split("_")[0][1:])
        y = int(base.split("_")[1][1:])
    except Exception:
        continue

    if x >= GRID_X or y >= GRID_Y:
        continue

    ms = base.split("_")[-1].replace(".npz", "")

    data = np.load(f, allow_pickle=True)

    if "neuron_ids" not in data or "responses" not in data:
        continue

    neuron_ids = data["neuron_ids"]
    responses = data["responses"]

    stim_dict = responses_by_ms.setdefault(ms, {}).setdefault((x, y), {})

    for nid, resp in zip(neuron_ids, responses):
        stim_dict[str(nid)] = resp


# ============================
# Compute ranked neurons separately for left and right
# ============================
top_neurons_by_ms_side = {}
top_100_neurons_by_ms_side = {}

for ms, stim_dict in responses_by_ms.items():

    peak_by_neuron_side = {
        "L": {},
        "R": {}
    }

    for (x, y), neuron_resp_dict in stim_dict.items():
        for nid, resp in neuron_resp_dict.items():

            nside = classification_df["side"].get(nid, "")

            if pd.isna(nside):
                continue

            nside = str(nside).upper()

            peak = np.max(resp)

            if nside.startswith("L"):
                peak_by_neuron_side["L"][nid] = max(
                    peak_by_neuron_side["L"].get(nid, 0),
                    peak
                )

            elif nside.startswith("R"):
                peak_by_neuron_side["R"][nid] = max(
                    peak_by_neuron_side["R"].get(nid, 0),
                    peak
                )

    for side in ["L", "R"]:
        ranked_neurons = sorted(
            peak_by_neuron_side[side],
            key=lambda nid: peak_by_neuron_side[side][nid],
            reverse=True
        )

        top_neurons_by_ms_side[(ms, side)] = ranked_neurons
        top_100_neurons_by_ms_side[(ms, side)] = ranked_neurons[:TOP_N]

# ============================
# Save top 100 neuron IDs
# ============================
for (ms, side), neuron_ids in top_100_neurons_by_ms_side.items():

    out_file = os.path.join(
        summary_dir,
        f"top100_neurons_{side}_{ms}.txt"
    )

    with open(out_file, "w") as f:
        for nid in neuron_ids:
            f.write(f"{nid},\n")

    print(f"{ms} {side} top 100 neuron IDs saved ({len(neuron_ids)} neurons).")


# ============================
# Prepare traces for one side
# ============================
def prepare_peak_trace_data(stim_dict, neuron_list, max_rows=TOP_K):
    traces = []
    peaks = []
    type_labels = []
    xy_labels = []

    for nid in neuron_list:

        best_peak = -np.inf
        best_trace = None
        best_xy = None

        for (x, y), neuron_resp_dict in stim_dict.items():

            if nid not in neuron_resp_dict:
                continue

            resp = neuron_resp_dict[nid]
            peak = np.max(resp)

            if peak > best_peak:
                best_peak = peak
                best_trace = resp
                best_xy = (x, y)

        if best_trace is None or best_peak <= 0:
            continue

        ntype = cell_type_df["primary_type"].get(nid)

        if ntype is None or pd.isna(ntype) or ntype == "Unknown":
            continue

        traces.append(best_trace)
        peaks.append(best_peak)
        type_labels.append(ntype)
        xy_labels.append(f"({best_xy[0]}, {best_xy[1]})")

        if len(traces) >= max_rows:
            break

    if len(traces) == 0:
        return None, None, None, None

    traces = np.array(traces)

    order = np.argsort(peaks)[::-1]

    traces = traces[order]
    peaks = [peaks[i] for i in order]
    type_labels = [type_labels[i] for i in order]
    xy_labels = [xy_labels[i] for i in order]

    if traces.shape[0] < max_rows:
        n_missing = max_rows - traces.shape[0]
        pad = np.zeros((n_missing, traces.shape[1]), dtype=traces.dtype)
        traces = np.vstack([traces, pad])
        peaks.extend([0] * n_missing)
        type_labels.extend([""] * n_missing)
        xy_labels.extend([""] * n_missing)

    return traces, peaks, type_labels, xy_labels

from matplotlib.ticker import NullLocator
from matplotlib.ticker import NullLocator

# ============================
# Plot L and R in one figure for each tau/ms
# ============================
def plot_peak_trace_summary_LR(ms, stim_dict):
    side_data = {}

    for side in ["L", "R"]:

        neuron_list = top_neurons_by_ms_side.get((ms, side), [])

        traces, peaks, type_labels, xy_labels = prepare_peak_trace_data(
            stim_dict,
            neuron_list,
            max_rows=TOP_K
        )

        side_data[side] = {
            "traces": traces,
            "peaks": peaks,
            "type_labels": type_labels,
            "xy_labels": xy_labels
        }

    valid_traces = [
        side_data[side]["traces"]
        for side in ["L", "R"]
        if side_data[side]["traces"] is not None
    ]

    if len(valid_traces) == 0:
        return

    global_vmax = max(np.max(np.abs(tr)) for tr in valid_traces)

    if global_vmax == 0:
        return

    norm = TwoSlopeNorm(
        vmin=-global_vmax,
        vcenter=0.0,
        vmax=global_vmax
    )

    colors = ["#2244bb", "#88aaff", "#ffffff", "#ffaaaa", "#bb2222"]
    cmap = LinearSegmentedColormap.from_list("soft_rwb", colors, N=256)

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(12, 16),
        sharex=False,
        sharey=False
    )

    fig.subplots_adjust(wspace=0.5)

    im_for_cbar = None

    side_title_map = {
        "L": "Left",
        "R": "Right"
    }

    for ax, side in zip(axes, ["L", "R"]):

        traces = side_data[side]["traces"]
        type_labels = side_data[side]["type_labels"]

        if traces is None:
            ax.axis("off")
            ax.set_title(
                side_title_map[side],
                y=1.04,
                fontweight="normal",
                bbox=dict(
                    boxstyle="round,pad=0.22",
                    facecolor="white",
                    edgecolor="black",
                    linewidth=1.2
                )
            )
            continue

        im = ax.imshow(
            traces,
            aspect="auto",
            origin="lower",
            cmap=cmap,
            norm=norm
        )

        im_for_cbar = im

        n_time = traces.shape[1]

        ax.set_xlabel("Time (ms)")
        ax.set_title(
            side_title_map[side],
            y=1.02,                
            fontweight="normal",    
        )

        ax.set_ylim(-0.5, TOP_K - 0.5)
        ax.set_yticks(np.arange(TOP_K))
        ax.set_yticklabels(type_labels[:TOP_K])

        xticks = np.linspace(0, n_time - 1, 5, dtype=int)
        ax.set_xticks(xticks)
        ax.set_xticklabels([str(x * 10) for x in xticks])

        ax.tick_params(
            axis="x",
            which="both",
            top=False,
            labeltop=False,
            bottom=True,
            labelbottom=True
        )
        ax.tick_params(
            axis="y",
            which="both",
            right=False,
            labelright=False,
            pad=8
        )

        ax.xaxis.set_minor_locator(NullLocator())
        ax.yaxis.set_minor_locator(NullLocator())
        ax.minorticks_off()

        for tick, label in zip(ax.get_yticklabels(), type_labels[:TOP_K]):
            tick.set_fontweight("normal")
            if label in HIGHLIGHT_TYPES:
                tick.set_bbox(dict(
                    boxstyle="round,pad=0.18",
                    facecolor="white",
                    edgecolor="black",
                    linewidth=1.0
                ))

    if im_for_cbar is not None:
        cbar = fig.colorbar(
            im_for_cbar,
            ax=axes,
            fraction=0.035,
            pad=0.045
        )
        cbar.ax.tick_params(labelsize=24)
        cbar.ax.yaxis.set_minor_locator(NullLocator())
        cbar.ax.minorticks_off()
        cbar.set_label(
            "PRS",
            fontsize=28,
            labelpad=12
        )

    fig.savefig(
        os.path.join(summary_dir, f"summary_peak_trace_Left_Right_{ms}.png"),
        dpi=600,
        bbox_inches="tight"
    )

    plt.close(fig)


# ============================
# Plot heatmaps for all tau/ms
# ============================
print("Start plotting Left/Right summary heatmaps...")

for ms, stim_dict in responses_by_ms.items():
    plot_peak_trace_summary_LR(ms, stim_dict)

print("\n✅ Summary heatmaps finished and top 100 neuron IDs saved.")

Loading response files...


100%|██████████| 3162/3162 [00:12<00:00, 249.74it/s]


80ms L top 100 neuron IDs saved (100 neurons).
80ms R top 100 neuron IDs saved (100 neurons).
20ms L top 100 neuron IDs saved (100 neurons).
20ms R top 100 neuron IDs saved (100 neurons).
Start plotting Left/Right summary heatmaps...

✅ Summary heatmaps finished and top 100 neuron IDs saved.


# Fig. 4(b)

In [1]:
import os
import glob
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from tqdm import tqdm
import scienceplots

# =====================================================
# STYLE
# =====================================================
plt.style.use(['science', 'nature', 'no-latex'])

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans"],
    "font.size": 20,
    "axes.labelsize": 20,
    "axes.titlesize": 20,
    "legend.fontsize": 18,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "axes.linewidth": 1.5,
    "figure.dpi": 300,
    "savefig.dpi": 600,
    "xtick.top": False,
    "ytick.right": False,
    "xtick.minor.visible": False,
    "ytick.minor.visible": False,
})

# =====================================================
# CONFIG
# =====================================================
RESPONSES_DIR = "../results/PRS/looming/202606/"
OUT_DIR = "../results/Figure4/"
os.makedirs(OUT_DIR, exist_ok=True)

CELL_TYPE_FILE = "../data/cell_type.txt"
CLASSIFICATION_FILE = "../data/classification.txt"
CONNECTION_FILE = "../data/connections.txt"

# 5 rows x 6 cols: DNp01, DNp02, DNp03, DNp04, DNp11
TARGET_TYPES = ["DNp01", "DNp02", "DNp03", "DNp04", "DNp11"]
TARGET_MS = ["20ms", "80ms"]
SIDE_ORDER = ["left", "right"]

# Predicted response: left/right use the same color
RESPONSE_COLOR = "#1E3A8A"
RV_COLOR = "#808080"
CROSS_COLOR = "red"

# single-layer donut chart: top upstream types, others merged
# ring: neuron count proportion
TOP_N_UPSTREAM_TYPES = 10
PIE_ALPHA = 0.8
OUTER_DONUT_WIDTH = 0.28
TYPE_COLORS = [
    "#102461",
    "#2576bc",
    "#7dc9c9",
    "#b7d79a",
    "#8da0cb",
    "#f28e2b",
    "#d95f02",
    "#8B0000",
    "#984ea3",
    "#4daf4a",
    "#66c2a5",
    "#fc8d62",
    "#8da0cb",
    "#e78ac3",
    "#a6d854",
    "#ffd92f",
    "#e5c494",
]
OTHER_COLOR = "#B0B0B0"
FONT_SIZE = 28

# mark missing/blocked condition
CROSS_RULES = [
    {"neuron_type": "DNp03", "side": "right"},
]

# =====================================================
# TABLE UTILS
# =====================================================
def read_table_auto(path, header="infer", names=None, dtype=None):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")

    # Try comma, tab, then python sniffing.
    for kwargs in [
        dict(sep=",", header=header, names=names, dtype=dtype),
        dict(sep="\t", header=header, names=names, dtype=dtype),
        dict(sep=None, engine="python", header=header, names=names, dtype=dtype),
    ]:
        try:
            df = pd.read_csv(path, **kwargs)
            if df.shape[1] > 1:
                return df
        except Exception:
            pass

    return pd.read_csv(path, header=header, names=names, dtype=dtype)


def normalize_id_series(s):
    return (
        s.astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
    )


def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    if x.lower() in ["nan", "none", "null"]:
        return ""
    return x


def clean_type_name(x):
    x = clean_text(x)
    return x if x else "Unknown"


def load_cell_type(path):
    df = read_table_auto(path, header=None, names=["root_id", "primary_type", "additional_type"], dtype=str)
    df["root_id"] = normalize_id_series(df["root_id"])
    df["primary_type"] = df["primary_type"].apply(clean_type_name)
    df = df.drop_duplicates(subset=["root_id"])
    df = df.set_index("root_id")
    return df


def load_classification(path):
    df = read_table_auto(path, dtype={"root_id": str})
    if "root_id" not in df.columns or "side" not in df.columns:
        raise ValueError("classification file must contain columns: root_id, side")
    df["root_id"] = normalize_id_series(df["root_id"])
    df["side"] = df["side"].astype(str).str.lower().str.strip()
    df = df.drop_duplicates(subset=["root_id"]).set_index("root_id")
    return df


def load_connections(path):
    conn = read_table_auto(path, dtype=str)
    required_cols = {"pre_root_id", "post_root_id"}
    missing = required_cols - set(conn.columns)
    if missing:
        raise ValueError(f"connections file missing columns: {missing}")

    conn["pre_root_id"] = normalize_id_series(conn["pre_root_id"])
    conn["post_root_id"] = normalize_id_series(conn["post_root_id"])

    if "syn_count" in conn.columns:
        conn["syn_count"] = pd.to_numeric(conn["syn_count"], errors="coerce").fillna(0)
    else:
        conn["syn_count"] = 1

    return conn

# =====================================================
# LOOM RESPONSE LOADING
# =====================================================
def ms_sort_key(ms):
    try:
        return float(str(ms).replace("ms", ""))
    except Exception:
        return str(ms)


def load_responses(responses_dir):
    all_files = glob.glob(os.path.join(responses_dir, "*.npz"))
    responses_by_ms = {}

    print("Loading response files...")
    for f in tqdm(all_files):
        base = os.path.basename(f)
        parts = base.split("_")

        try:
            x = int(parts[0][1:])
            y = int(parts[1][1:])
        except Exception:
            continue

        ms = parts[-1].replace(".npz", "")
        data = np.load(f, allow_pickle=True)

        if "neuron_ids" not in data or "responses" not in data:
            continue

        neuron_ids = data["neuron_ids"].astype(str)
        responses = data["responses"]

        stim_dict = responses_by_ms.setdefault(ms, {}).setdefault((x, y), {})
        for nid, resp in zip(neuron_ids, responses):
            stim_dict[nid] = resp

    return responses_by_ms


def build_neurons_by_type(responses_by_ms, cell_type_df):
    neurons_by_type = {t: [] for t in TARGET_TYPES}

    for _, stim_dict in responses_by_ms.items():
        neuron_set = set()
        for neuron_resp_dict in stim_dict.values():
            neuron_set.update(neuron_resp_dict.keys())

        for t in TARGET_TYPES:
            neurons_by_type[t].extend([
                nid for nid in neuron_set
                if cell_type_df["primary_type"].get(nid, "Unknown") == t
            ])

    for t in TARGET_TYPES:
        neurons_by_type[t] = sorted(set(neurons_by_type[t]))

    return neurons_by_type

# =====================================================
# UPSTREAM PIE DATA
# =====================================================
def build_upstream_donut_pivots(conn, cell_type_df, classification_df):
    """
    Build upstream neuron-count distribution for each DN type and side.

    ring: upstream neuron count proportion
        each pre_root_id is counted once within each DN_type + DN_side + upstream_type
    """
    celltype_reset = cell_type_df.reset_index()[["root_id", "primary_type"]]

    dn_table = celltype_reset[celltype_reset["primary_type"].isin(TARGET_TYPES)].copy()
    side_map = classification_df.reset_index()[["root_id", "side"]]
    dn_table = dn_table.merge(side_map, how="left", on="root_id")
    dn_table["side"] = dn_table["side"].astype(str).str.lower().str.strip()

    dn_info = dn_table[["root_id", "primary_type", "side"]].rename(columns={
        "root_id": "post_root_id",
        "primary_type": "DN_type",
        "side": "DN_side",
    })

    work = conn[conn["post_root_id"].isin(dn_info["post_root_id"].astype(str))].copy()
    work = work.merge(dn_info, how="left", on="post_root_id")

    pre_type_map = celltype_reset.rename(columns={
        "root_id": "pre_root_id",
        "primary_type": "upstream_type",
    })
    work = work.merge(pre_type_map, how="left", on="pre_root_id")
    work["upstream_type"] = work["upstream_type"].apply(clean_type_name)
    work["syn_count"] = pd.to_numeric(work["syn_count"], errors="coerce").fillna(0)
    # remove Unknown neurons
    work = work[
        (work["upstream_type"] != "Unknown") &
        (work["upstream_type"] != "") &
        (~work["upstream_type"].isna())
    ].copy()
    # outer ring: neuron count
    work_unique = work.drop_duplicates(
        subset=["DN_type", "DN_side", "pre_root_id", "upstream_type"]
    )
    count_dist_raw = (
        work_unique
        .groupby(["DN_type", "DN_side", "upstream_type"], as_index=False)
        .size()
        .rename(columns={"size": "value"})
    )

    # global top upstream types based on neuron count; remaining types are merged as Other
    totals = count_dist_raw.groupby("upstream_type")["value"].sum().sort_values(ascending=False)
    top_types = totals.head(TOP_N_UPSTREAM_TYPES).index.tolist()

    def merge_to_top(df):
        df = df.copy()
        df["plot_type"] = np.where(df["upstream_type"].isin(top_types), df["upstream_type"], "Other")
        return (
            df
            .groupby(["DN_type", "DN_side", "plot_type"], as_index=False)["value"]
            .sum()
        )

    count_data = merge_to_top(count_dist_raw)
    count_totals = (
        count_dist_raw
        .groupby(["DN_type", "DN_side"], as_index=False)["value"]
        .sum()
        .rename(columns={"value": "neuron_count"})
    )
    total_data = count_totals.fillna(0)

    type_order = [t for t in top_types if (count_data["plot_type"] == t).any()]
    if (count_data["plot_type"] == "Other").any():
        type_order.append("Other")

    return count_data, total_data, type_order

# =====================================================
# RV CURVE
# =====================================================
def generate_rv_curve(
    tau,
    r_start_px=4,
    r_max_px=20,
    frame_rate=1000,
    gray_start_sec=0.6,
    gray_end_sec=0.5,
    ppd=1.0
):
    angle_max_rad = np.deg2rad(r_max_px / ppd)
    angle_start_rad = np.deg2rad(r_start_px / ppd)

    t_min = tau / np.tan(angle_max_rad)
    t_max = tau / np.tan(angle_start_rad)
    looming_duration_sec = t_max - t_min
    looming_frames = int(np.ceil(looming_duration_sec * frame_rate))

    gray_start_frames = int(gray_start_sec * frame_rate)
    gray_end_frames = int(gray_end_sec * frame_rate)
    total_frames = gray_start_frames + looming_frames + gray_end_frames

    rv_frames = np.zeros(total_frames)

    for f in range(looming_frames):
        t_elapsed = f / frame_rate
        t_remain = max(t_max - t_elapsed, 1e-6)
        angle_rad = np.arctan(tau / t_remain)
        rv_frames[gray_start_frames + f] = np.rad2deg(angle_rad)

    rv_frames[gray_start_frames + looming_frames:] = np.rad2deg(angle_max_rad)

    if len(rv_frames) > 500:
        return rv_frames[500::10]
    return rv_frames[::10]

# =====================================================
# PLOTTING HELPERS
# =====================================================
def is_cross_cell(neuron_type, side):
    for rule in CROSS_RULES:
        if rule.get("neuron_type") == neuron_type and rule.get("side") == side:
            return True
    return False


def build_pie_color_map(type_order):
    color_map = {}
    color_idx = 0
    for t in type_order:
        if t == "Other":
            color_map[t] = OTHER_COLOR
        else:
            color_map[t] = TYPE_COLORS[color_idx % len(TYPE_COLORS)]
            color_idx += 1
    return color_map


def plot_upstream_donut(ax, count_data, total_data, type_order, neuron_type, side):
    ax.axis("off")

    count_sub = count_data[(count_data["DN_type"] == neuron_type) & (count_data["DN_side"] == side)]

    if count_sub.empty:
        ax.text(0.5, 0.5, "NA", ha="center", va="center", fontsize=FONT_SIZE)
        return

    color_map = build_pie_color_map(type_order)

    values = []
    labels = []
    for t in type_order:
        v = count_sub.loc[count_sub["plot_type"] == t, "value"].sum()
        if v > 0:
            values.append(v)
            labels.append(t)

    colors = [color_map.get(label, OTHER_COLOR) for label in labels]

    if values:
        ax.pie(
            values,
            radius=1,
            colors=colors,
            startangle=90,
            counterclock=False,
            wedgeprops={
                "width": OUTER_DONUT_WIDTH,
                "linewidth": 0.5,
                "edgecolor": "white",
                "alpha": PIE_ALPHA,
            },
        )

    total_sub = total_data[(total_data["DN_type"] == neuron_type) & (total_data["DN_side"] == side)]
    if total_sub.empty:
        neuron_count = 0
    else:
        neuron_count = int(round(float(total_sub["neuron_count"].iloc[0])))

    ax.text(
        0,
        0,
        f"N={neuron_count}",
        ha="center",
        va="center",
        fontsize=22,
        linespacing=1.05,
    )
    ax.set_aspect("equal")

def normalize_to_0_1(arr):
    arr = np.asarray(arr, dtype=float)
    finite = np.isfinite(arr)
    if not np.any(finite):
        return np.zeros_like(arr, dtype=float)

    arr_min = np.nanmin(arr[finite])
    arr_max = np.nanmax(arr[finite])

    if np.isclose(arr_max, arr_min):
        return np.zeros_like(arr, dtype=float)

    out = (arr - arr_min) / (arr_max - arr_min)
    out[~np.isfinite(out)] = 0
    return out


def plot_trace(ax, responses_by_ms, neurons_by_type, classification_df, neuron_type, side, ms):
    ax.axis("off")

    if ms not in responses_by_ms:
        ax.text(0.5, 0.5, "NA", ha="center", va="center", fontsize=FONT_SIZE)
        return

    ms_num = str(ms).replace("ms", "")
    tau_val = float(ms_num) / 1000.0
    rv_curve = generate_rv_curve(tau_val)
    stim_dict = responses_by_ms[ms]

    neurons = [
        nid for nid in neurons_by_type.get(neuron_type, [])
        if classification_df["side"].get(nid, "Unknown") == side
    ]

    if not neurons:
        ax.text(0.5, 0.5, "NA", ha="center", va="center", fontsize=FONT_SIZE)
        return

    max_resp = None
    max_val = -np.inf

    for _, neuron_resp_dict in stim_dict.items():
        for nid in neurons:
            if nid not in neuron_resp_dict:
                continue
            resp = np.asarray(neuron_resp_dict[nid], dtype=float)
            peak = np.nanmax(resp)
            if peak > max_val:
                max_val = peak
                max_resp = resp

    if max_resp is None or not np.isfinite(max_val):
        ax.text(0.5, 0.5, "NA", ha="center", va="center", fontsize=FONT_SIZE)
        return

    # y_norm = (y - y.min()) / (y.max() - y.min())
    max_resp_norm = normalize_to_0_1(max_resp)
    rv_curve_norm = normalize_to_0_1(rv_curve)

    if len(rv_curve_norm) < len(max_resp_norm):
        rv_curve_norm = np.pad(
            rv_curve_norm,
            (0, len(max_resp_norm) - len(rv_curve_norm)),
            mode="edge"
        )
    else:
        rv_curve_norm = rv_curve_norm[:len(max_resp_norm)]

    ax.plot(
        max_resp_norm,
        color=RESPONSE_COLOR,
        linewidth=2.2
    )

    ax.plot(
        rv_curve_norm,
        color=RV_COLOR,
        linestyle="--",
        linewidth=1.6
    )

    ax.set_ylim(-0.05, 1.05)
    ax.set_xlim(0, len(max_resp_norm) - 1)

    if is_cross_cell(neuron_type, side):
        ax.plot(
            0.18,
            0.82,
            marker="x",
            color=CROSS_COLOR,
            markersize=12,
            markeredgewidth=2.2,
            transform=ax.transAxes,
            linestyle="None"
        )

# =====================================================
# MAIN FIGURE
# =====================================================
def plot_4x6_loom_donut_trace(responses_by_ms, neurons_by_type, count_data, total_data, type_order, classification_df):
    column_order = [
        ("left", "pie"),
        ("left", "20ms"),
        ("left", "80ms"),
        ("right", "pie"),
        ("right", "20ms"),
        ("right", "80ms"),
    ]

    n_rows = len(TARGET_TYPES)
    n_cols = len(column_order)

    # Custom layout:
    # Left region: 3 columns
    # Middle gap: 0.15 relative width
    # Right region: 3 columns
    fig = plt.figure(
        figsize=(22, max(11, n_rows * 2.6))
    )

    gs = fig.add_gridspec(
        n_rows,
        7,
        width_ratios=[
            1,      # Left donut
            1,      # Left 20 ms
            1,      # Left 80 ms
            0.15,   # Gap between Left and Right regions
            1,      # Right donut
            1,      # Right 20 ms
            1       # Right 80 ms
        ]
    )

    axes = np.empty((n_rows, 6), dtype=object)

    for r in range(n_rows):
        axes[r, 0] = fig.add_subplot(gs[r, 0])
        axes[r, 1] = fig.add_subplot(gs[r, 1])
        axes[r, 2] = fig.add_subplot(gs[r, 2])

        # gs[r, 3] is the blank gap column

        axes[r, 3] = fig.add_subplot(gs[r, 4])
        axes[r, 4] = fig.add_subplot(gs[r, 5])
        axes[r, 5] = fig.add_subplot(gs[r, 6])

    fig.subplots_adjust(
        left=0.08,
        right=0.82,
        bottom=0.06,
        top=0.88,
        wspace=0.22,
        hspace=0.18
    )

    for row_idx, neuron_type in enumerate(TARGET_TYPES):
        # row label
        axes[row_idx, 0].text(
            -0.1,
            0.5,
            neuron_type,
            transform=axes[row_idx, 0].transAxes,
            ha="center",
            va="center",
            rotation=90,
            fontsize=22
        )

        for col_idx, (side, item) in enumerate(column_order):
            ax = axes[row_idx, col_idx]
            if item == "pie":
                plot_upstream_donut(ax, count_data, total_data, type_order, neuron_type, side)
            else:
                plot_trace(ax, responses_by_ms, neurons_by_type, classification_df, neuron_type, side, item)

    # column titles
    titles = ["", "20 ms", "80 ms", "", "20 ms", "80 ms"]
    for col_idx, title in enumerate(titles):
        if title:
            axes[0, col_idx].set_title(title, fontsize=FONT_SIZE, pad=8)

    # side labels in the center of every 3 columns
    left_center = (axes[0, 0].get_position().x0 + axes[0, 2].get_position().x1) / 2
    right_center = (axes[0, 3].get_position().x0 + axes[0, 5].get_position().x1) / 2

    fig.text(left_center, 0.94, "Left", ha="center", va="center", fontsize=FONT_SIZE)
    fig.text(right_center, 0.94, "Right", ha="center", va="center", fontsize=FONT_SIZE)

    # shared legends on the right side
    # Type legend: pie-chart color patches
    pie_color_map = build_pie_color_map(type_order)

    type_handles = [
        Patch(
            facecolor=pie_color_map.get(t, OTHER_COLOR),
            edgecolor="none",
            alpha=PIE_ALPHA,
            label=t
        )
        for t in type_order
    ]

    type_legend = fig.legend(
        handles=type_handles,
        title="Neuron type",
        loc="upper left",
        bbox_to_anchor=(0.84, 0.82),
        frameon=False,
        fontsize=20,
        title_fontsize=20,
        handlelength=1.6,
        borderaxespad=0.0,
        labelspacing=0.65
    )
    fig.add_artist(type_legend)

    # Predicted response legend: left/right are not separated
    response_handles = [
        Line2D([0], [0], color=RESPONSE_COLOR, lw=2.4, label="PRC"),
        Line2D([0], [0], color=RV_COLOR, lw=2.0, ls="--", label="r/v"),
        Line2D([0], [0], color=CROSS_COLOR, marker="x", linestyle="None", markersize=11,
               markeredgewidth=2.2, label="Missing"),
    ]

    fig.legend(
        handles=response_handles,
        loc="upper left",
        title=" ",
        bbox_to_anchor=(0.84, 0.3),
        frameon=False,
        fontsize=20,
        title_fontsize=20,
        handlelength=2.2,
        borderaxespad=0.0,
        labelspacing=0.65
    )

    out_png = os.path.join(OUT_DIR, "loom_5x6_upstream_top5_single_donut_20_80_single_color_rv_individual_norm01.png")

    fig.savefig(out_png, dpi=600, bbox_inches="tight")
    plt.close(fig)
    gc.collect()

    print("\n[saved]")
    print(out_png)

# =====================================================
# RUN
# =====================================================
if __name__ == "__main__":
    cell_type_df = load_cell_type(CELL_TYPE_FILE)
    classification_df = load_classification(CLASSIFICATION_FILE)
    conn = load_connections(CONNECTION_FILE)

    responses_by_ms = load_responses(RESPONSES_DIR)

    # Only keep 20ms and 80ms for the requested 4x6 layout.
    responses_by_ms = {
        ms: responses_by_ms[ms]
        for ms in TARGET_MS
        if ms in responses_by_ms
    }

    missing_ms = [ms for ms in TARGET_MS if ms not in responses_by_ms]
    if missing_ms:
        print(f"[warning] Missing response ms files: {missing_ms}")

    neurons_by_type = build_neurons_by_type(responses_by_ms, cell_type_df)
    count_data, total_data, type_order = build_upstream_donut_pivots(conn, cell_type_df, classification_df)

    plot_4x6_loom_donut_trace(
        responses_by_ms=responses_by_ms,
        neurons_by_type=neurons_by_type,
        count_data=count_data,
        total_data=total_data,
        type_order=type_order,
        classification_df=classification_df
    )

Loading response files...


100%|██████████| 3162/3162 [00:13<00:00, 232.72it/s]



[saved]
../results/Figure4/loom_5x6_upstream_top5_single_donut_20_80_single_color_rv_individual_norm01.png


In [1]:
import os
import glob
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm
from matplotlib.colors import Normalize
import scienceplots

# -----------------------------
# 1. STYLE & PATH CONFIG
# -----------------------------
plt.style.use(['science', 'nature', 'no-latex'])

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans"],
    "axes.linewidth": 1.5,
    "figure.dpi": 600,
    "savefig.dpi": 600,
})

# Input response files
responses_dir = "../results/PRS/looming/202606/"

# Only output one big pure heatmap.
# Output directory and file names are defined after TARGET_GRID_X below.

cell_type_file = "../data/cell_type.txt"
classification_file = "../data/classification.txt"

# -----------------------------
# 2. BASIC CONFIG
# -----------------------------
GRID_Y = 41
GRID_X = 82

# Target heatmap width after removing the middle columns.
# Change this value only if you want another output width.
TARGET_GRID_X = 77

REMOVE_MIDDLE_N_COLS = GRID_X - TARGET_GRID_X
if REMOVE_MIDDLE_N_COLS < 0:
    raise ValueError("TARGET_GRID_X cannot be larger than GRID_X.")
if REMOVE_MIDDLE_N_COLS >= GRID_X:
    raise ValueError("TARGET_GRID_X must be positive.")

REMOVE_START_COL = (GRID_X - REMOVE_MIDDLE_N_COLS) // 2
REMOVE_END_COL = REMOVE_START_COL + REMOVE_MIDDLE_N_COLS
FINAL_GRID_X = TARGET_GRID_X

out_dir = f"../results/Figure4/"
os.makedirs(out_dir, exist_ok=True)
out_png = os.path.join(out_dir, f"c.png")

TARGET_TYPES_ORDER = ["DNp01", "DNp02", "DNp03", "DNp04", "DNp11"]
PANEL_ORDER = [
    ("left", "20ms"),
    ("left", "80ms"),
    ("right", "20ms"),
    ("right", "80ms"),
]

SIDE_ALIASES = {
    "left": "left",
    "l": "left",
    "right": "right",
    "r": "right",
}

MS_ALIASES = {
    "20": "20ms",
    "20ms": "20ms",
    "80": "80ms",
    "80ms": "80ms",
}

# -----------------------------
# 3. LOAD METADATA
# -----------------------------
cell_type_df = pd.read_csv(
    cell_type_file,
    header=None,
    names=["root_id", "primary_type", "additional_type"],
    dtype={"root_id": str}
)
cell_type_df.set_index("root_id", inplace=True)
cell_type_df.index = cell_type_df.index.astype(str)

classification_df = pd.read_csv(
    classification_file,
    dtype={"root_id": str}
)
classification_df.set_index("root_id", inplace=True)
classification_df.index = classification_df.index.astype(str)


def normalize_side(side):
    side = str(side).strip().lower()
    return SIDE_ALIASES.get(side, side)


def normalize_ms(ms):
    ms = str(ms).strip().lower().replace("_", "")
    return MS_ALIASES.get(ms, ms)


def crop_middle_cols(peak_map):
    """
    Original peak_map shape: GRID_Y x GRID_X.
    Remove middle columns automatically to reach TARGET_GRID_X.
    """
    return np.delete(
        peak_map,
        np.arange(REMOVE_START_COL, REMOVE_END_COL),
        axis=1
    )

# -----------------------------
# 4. LOAD RESPONSE FILES
# -----------------------------
all_files = glob.glob(os.path.join(responses_dir, "*.npz"))
responses_by_ms = {}

print("Loading response files...")
for f in tqdm(all_files):
    base = os.path.basename(f)

    try:
        x = int(base.split("_")[0][1:])
        y = int(base.split("_")[1][1:])
    except Exception:
        continue

    if x >= GRID_X or y >= GRID_Y:
        continue

    raw_ms = base.split("_")[-1].replace(".npz", "")
    ms = normalize_ms(raw_ms)

    data = np.load(f, allow_pickle=True)
    if "neuron_ids" not in data or "responses" not in data:
        continue

    neuron_ids = data["neuron_ids"].astype(str)
    responses = data["responses"]

    stim_dict = responses_by_ms.setdefault(ms, {}).setdefault((x, y), {})
    for nid, resp in zip(neuron_ids, responses):
        stim_dict[nid] = resp

# -----------------------------
# 5. BUILD HEATMAPS BY TYPE / SIDE / MS
# -----------------------------
def make_peak_map_for_neuron(neuron_id, stim_dict):
    peak_map = np.full((GRID_Y, GRID_X), np.nan, dtype=np.float32)

    for (x, y), neuron_resp_dict in stim_dict.items():
        if neuron_id not in neuron_resp_dict:
            continue

        resp = np.asarray(neuron_resp_dict[neuron_id], dtype=np.float32)
        if resp.size == 0 or np.all(~np.isfinite(resp)):
            continue

        peak_map[y, x] = max(float(np.nanmax(resp)), 0.0)

    if np.all(np.isnan(peak_map)):
        return None

    return peak_map


def make_group_peak_map(neuron_type, side, ms):
    """
    If multiple neurons exist for the same DN type / side / ms,
    average them into one panel.
    """
    if ms not in responses_by_ms:
        return None

    stim_dict = responses_by_ms[ms]
    maps = []

    neuron_set = set()
    for neuron_resp_dict in stim_dict.values():
        neuron_set.update(neuron_resp_dict.keys())

    for nid in neuron_set:
        this_type = cell_type_df["primary_type"].get(nid, "Unknown")
        this_side = normalize_side(classification_df["side"].get(nid, "Unknown"))

        if this_type != neuron_type or this_side != side:
            continue

        peak_map = make_peak_map_for_neuron(nid, stim_dict)
        if peak_map is not None:
            maps.append(peak_map)

    if len(maps) == 0:
        return None

    stacked = np.stack(maps, axis=0)
    group_map = np.nanmean(stacked, axis=0)
    return crop_middle_cols(group_map)


panel_maps = {}
for neuron_type in TARGET_TYPES_ORDER:
    for side, ms in PANEL_ORDER:
        panel_maps[(neuron_type, side, ms)] = make_group_peak_map(neuron_type, side, ms)

has_data = any(
    m is not None and np.any(np.isfinite(m))
    for m in panel_maps.values()
)
if not has_data:
    raise RuntimeError(
        "No valid heatmap data found. Please check responses_dir, "
        "cell_type_file, and classification_file."
    )

# -----------------------------
# 6. PLOT ONE BIG PURE HEATMAP
# -----------------------------
# Layout rule:
#   left-20ms and left-80ms: tightly attached
#   right-20ms and right-80ms: tightly attached
#   gap between left group and right group: 0.15
#
# Change GROUP_WSPACE if you want to adjust only the left/right group gap.
GROUP_WSPACE = 0.05

fig = plt.figure(figsize=(24, 16))  # height increased by 4: 10 -> 14
outer_gs = fig.add_gridspec(
    nrows=1,
    ncols=2,
    left=0,
    right=1,
    top=1,
    bottom=0,
    wspace=GROUP_WSPACE,
)

left_gs = outer_gs[0, 0].subgridspec(
    nrows=len(TARGET_TYPES_ORDER),
    ncols=2,
    wspace=0.0,
    hspace=0.0,
)
right_gs = outer_gs[0, 1].subgridspec(
    nrows=len(TARGET_TYPES_ORDER),
    ncols=2,
    wspace=0.0,
    hspace=0.0,
)

axes = np.empty((len(TARGET_TYPES_ORDER), len(PANEL_ORDER)), dtype=object)
for i in range(len(TARGET_TYPES_ORDER)):
    axes[i, 0] = fig.add_subplot(left_gs[i, 0])   # left 20ms
    axes[i, 1] = fig.add_subplot(left_gs[i, 1])   # left 80ms
    axes[i, 2] = fig.add_subplot(right_gs[i, 0])  # right 20ms
    axes[i, 3] = fig.add_subplot(right_gs[i, 1])  # right 80ms

cmap = plt.cm.magma

for i, neuron_type in enumerate(TARGET_TYPES_ORDER):
    for j, (side, ms) in enumerate(PANEL_ORDER):
        ax = axes[i, j]
        ax.axis("off")

        peak_map = panel_maps[(neuron_type, side, ms)]
        if peak_map is None or not np.any(np.isfinite(peak_map)):
            ax.imshow(
                np.zeros((GRID_Y, FINAL_GRID_X)),
                origin="lower",
                cmap=cmap,
                norm=Normalize(vmin=0, vmax=1),
                aspect="auto",
                alpha=0.0,
                interpolation="nearest",
            )
            continue

        # Original heatmap normalization style:
        # each panel uses its own vmax, with color range from 0.5*vmax to vmax.
        local_vmax = float(np.nanmax(peak_map))
        if not np.isfinite(local_vmax) or local_vmax <= 0:
            local_vmax = 1.0

        ax.imshow(
            peak_map,
            origin="lower",
            cmap=cmap,
            norm=Normalize(vmin=0.5 * local_vmax, vmax=local_vmax),
            aspect="auto",
            interpolation="nearest",
        )

# Pure image: no labels, no title, no colorbar, no margins
fig.savefig(out_png, dpi=600, bbox_inches="tight", pad_inches=0)
plt.close(fig)

gc.collect()
print(f"Saved: {out_png}")


Loading response files...


100%|██████████| 3162/3162 [00:13<00:00, 235.23it/s]
/tmp/ipykernel_69300/3819529762.py:206: RuntimeWarning: Mean of empty slice
  group_map = np.nanmean(stacked, axis=0)


Saved: ../results/Figure4/c.png


In [6]:
import os
import glob
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import scienceplots
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

plt.style.use(['science', 'nature', 'no-latex'])

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans"],
    "font.size": 30,
    "axes.labelsize": 30,
    "axes.titlesize": 30,
    "legend.fontsize": 30,
    "xtick.labelsize": 30,
    "ytick.labelsize": 30,
    "axes.linewidth": 2.0,
})

# =========================
# PATHS
# =========================
RESPONSES_DIR_CTRL      = "../results/PRS/looming/202606/"
RESPONSES_DIR_LC4       = "../results/PRS/looming_LC4/202606/"
RESPONSES_DIR_LPLC2     = "../results/PRS/looming_LPLC2/202606/"
RESPONSES_DIR_LC4_LPLC2 = "../results/PRS/looming_LC4_LPLC2/202606/"

OUT_DIR = "../results/Figure4/"
os.makedirs(OUT_DIR, exist_ok=True)

cell_type_file = "../data/cell_type.txt"
classification_file = "../data/classification.txt"

TARGET_TYPE = "DNp01"
TARGET_MS = ["20ms", "80ms"]
PANEL_ORDER = [
    ("left", "20ms"),
    ("left", "80ms"),
    ("right", "20ms"),
    ("right", "80ms"),
]

BLOCK_ROWS = [
    ("LC4", RESPONSES_DIR_LC4, "#6E2DBB"),
    ("LPLC2", RESPONSES_DIR_LPLC2, "#6E2DBB"),
    ("LC4 & LPLC2", RESPONSES_DIR_LC4_LPLC2, "#6E2DBB"),
]

COLOR_CTRL = "#49A39B"
COLOR_BLOCK = "#6E2DBB"
COLOR_RV = "#666666"

# =========================
# LOAD METADATA
# =========================
cell_type_df = pd.read_csv(
    cell_type_file,
    header=None,
    names=["root_id", "primary_type", "additional_type"],
    dtype={"root_id": str}
)
cell_type_df.set_index("root_id", inplace=True)

classification_df = pd.read_csv(
    classification_file,
    dtype={"root_id": str}
)
classification_df.set_index("root_id", inplace=True)

# =========================
# LOAD RESPONSES
# =========================
def load_responses_by_ms(responses_dir):
    responses_by_ms = {}
    all_files = glob.glob(os.path.join(responses_dir, "*.npz"))

    for f in tqdm(all_files):
        base = os.path.basename(f)
        try:
            x = int(base.split("_")[0][1:])
            y = int(base.split("_")[1][1:])
        except Exception:
            continue
        ms = base.split("_")[-1].replace(".npz", "")
        if ms not in TARGET_MS:
            continue
        data = np.load(f, allow_pickle=True)
        if "neuron_ids" not in data or "responses" not in data:
            continue
        neuron_ids = data["neuron_ids"].astype(str)
        responses = data["responses"]
        stim_dict = responses_by_ms.setdefault(ms, {}).setdefault((x, y), {})
        for nid, resp in zip(neuron_ids, responses):
            stim_dict[nid] = np.asarray(resp, dtype=float)
    return responses_by_ms

responses_ctrl = load_responses_by_ms(RESPONSES_DIR_CTRL)
responses_lc4 = load_responses_by_ms(RESPONSES_DIR_LC4)
responses_lplc2 = load_responses_by_ms(RESPONSES_DIR_LPLC2)
responses_lc4_lplc2 = load_responses_by_ms(RESPONSES_DIR_LC4_LPLC2)

RESPONSES_BY_BLOCK = {
    "LC4": responses_lc4,
    "LPLC2": responses_lplc2,
    "LC4 & LPLC2": responses_lc4_lplc2,
}

# =========================
# TARGET NEURONS
# =========================
def get_target_neurons(responses_by_ms, neuron_type, side):
    neuron_set = set()
    for _, stim_dict in responses_by_ms.items():
        for neuron_resp_dict in stim_dict.values():
            neuron_set.update(neuron_resp_dict.keys())
    neurons = []
    for nid in neuron_set:
        n_type = cell_type_df["primary_type"].get(nid, "Unknown")
        n_side = classification_df["side"].get(nid, "Unknown")
        if n_type == neuron_type and n_side == side:
            neurons.append(nid)
    return sorted(list(set(neurons)))

# =========================
# R/V CURVE
# =========================
def generate_rv_curve(tau, r_start_px=4, r_max_px=20, frame_rate=1000, gray_start_sec=0.6, gray_end_sec=0.5, ppd=1.0):
    angle_max_rad = np.deg2rad(r_max_px / ppd)
    angle_start_rad = np.deg2rad(r_start_px / ppd)
    t_min = tau / np.tan(angle_max_rad)
    t_max = tau / np.tan(angle_start_rad)
    looming_duration_sec = t_max - t_min
    looming_frames = int(np.ceil(looming_duration_sec * frame_rate))
    gray_start_frames = int(gray_start_sec * frame_rate)
    gray_end_frames = int(gray_end_sec * frame_rate)
    total_frames = gray_start_frames + looming_frames + gray_end_frames
    rv_frames = np.zeros(total_frames)
    rv_frames[:gray_start_frames] = 0.0
    for f in range(looming_frames):
        t_elapsed = f / frame_rate
        t_remain = max(t_max - t_elapsed, 1e-6)
        angle_rad = np.arctan(tau / t_remain)
        rv_frames[gray_start_frames + f] = np.rad2deg(angle_rad)
    rv_frames[gray_start_frames + looming_frames:] = np.rad2deg(angle_max_rad)
    if len(rv_frames) > 500:
        rv_plot = rv_frames[500::10]
    else:
        rv_plot = rv_frames[::10]
    return rv_plot

# =========================
# CTRL PEAK POSITION + MATCHED TRACE
# =========================
def get_ctrl_peak_reference(responses_by_ms, ms, neurons):
    """
    Find the strongest response in CTRL for a given ms and target neuron set.

    Returns
    -------
    max_resp : np.ndarray or None
        CTRL response trace at the strongest peak.
    max_val : float
        Peak value of the selected CTRL trace.
    max_coord : tuple or None
        Looming stimulus coordinate (x, y) where the CTRL peak was found.
    max_nid : str or None
        Neuron id where the CTRL peak was found.
    """
    if ms not in responses_by_ms:
        return None, np.nan, None, None

    stim_dict = responses_by_ms[ms]

    max_resp = None
    max_val = -np.inf
    max_coord = None
    max_nid = None

    for coord, neuron_resp_dict in stim_dict.items():
        for nid in neurons:
            if nid not in neuron_resp_dict:
                continue

            resp = np.asarray(neuron_resp_dict[nid], dtype=float)
            peak = np.max(resp)

            if peak > max_val:
                max_val = peak
                max_resp = resp.copy()
                max_coord = coord
                max_nid = nid

    return max_resp, max_val, max_coord, max_nid


def get_trace_at_ctrl_peak_coord(
    responses_by_ms,
    ms,
    neurons,
    ctrl_peak_coord,
    preferred_nid=None
):
    """
    Get the response trace at the CTRL-selected looming coordinate.

    The blocked condition is NOT allowed to choose its own peak coordinate.
    It must use ctrl_peak_coord.

    Priority:
        1. Use the same neuron id as the CTRL peak if it exists.
        2. Otherwise, at the same coordinate, choose the strongest trace among
           the target neurons as a fallback.
    """
    if ms not in responses_by_ms:
        return None

    if ctrl_peak_coord is None:
        return None

    stim_dict = responses_by_ms[ms]

    if ctrl_peak_coord not in stim_dict:
        return None

    neuron_resp_dict = stim_dict[ctrl_peak_coord]

    # Prefer exactly the same neuron id selected in CTRL.
    if preferred_nid is not None and preferred_nid in neuron_resp_dict:
        return np.asarray(neuron_resp_dict[preferred_nid], dtype=float).copy()

    # Fallback: same coordinate, strongest target-neuron trace.
    max_resp = None
    max_val = -np.inf

    for nid in neurons:
        if nid not in neuron_resp_dict:
            continue

        resp = np.asarray(neuron_resp_dict[nid], dtype=float)
        peak = np.max(resp)

        if peak > max_val:
            max_val = peak
            max_resp = resp.copy()

    return max_resp

def normalize_to_0_1_each(trace):
    """
    Normalize one trace by its own min/max to 0-1.

    This is used for CTRL, LC4-blocked, and LPLC2-blocked curves.
    The double-blocked LC4 & LPLC2 curve is intentionally not normalized
    in the plotting function below.
    """
    if trace is None:
        return None

    trace = np.asarray(trace, dtype=float)
    trace_min = np.nanmin(trace)
    trace_max = np.nanmax(trace)
    denom = trace_max - trace_min

    if not np.isfinite(denom) or denom == 0:
        denom = 1.0

    return (trace - trace_min) / denom


def prepare_plot_traces(resp_ctrl, resp_block, block_label):
    """
    Prepare traces for plotting.

    Rules
    -----
    1. CTRL is normalized by its own min/max.
    2. LC4 and LPLC2 blocked traces are normalized by their own min/max.
    3. LC4 & LPLC2 blocked trace is kept as the original raw trace,
       so it will not be artificially stretched to 0-1.
    """
    resp_ctrl_plot = normalize_to_0_1_each(resp_ctrl)

    if resp_block is None:
        resp_block_plot = None
    elif block_label == "LC4 & LPLC2":
        # Do NOT normalize the double-blocked trace.
        resp_block_plot = np.asarray(resp_block, dtype=float).copy()
    else:
        # Normalize each single-blocked trace by its own min/max.
        resp_block_plot = normalize_to_0_1_each(resp_block)

    return resp_ctrl_plot, resp_block_plot

def match_length(arr, target_len):
    if arr is None:
        return None
    arr = np.asarray(arr, dtype=float)
    if len(arr) < target_len:
        arr = np.pad(arr, (0, target_len - len(arr)), mode="edge")
    else:
        arr = arr[:target_len]
    return arr

# =========================
# AXIS STYLE
# =========================
def style_axis_keep_xline(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.spines["bottom"].set_visible(True)
    ax.spines["bottom"].set_linewidth(2.0)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(axis="both", which="both", length=0)

# =========================
# PLOT FUNCTION
# =========================
def plot_dnp01_block_change_3x4_paper_style():
    fig, axes = plt.subplots(
        nrows=3,
        ncols=5,
        figsize=(24, 10),
        sharex=False,
        sharey=True,
        gridspec_kw={"width_ratios": [1.0, 1.0, 0.15, 1.0, 1.0], "wspace":0.12}
    )
    fig.subplots_adjust(left=0.04, right=0.88, bottom=0.06, top=0.98, hspace=0.22)
    col_map = [0,1,3,4]
    for row_idx in range(3):
        axes[row_idx, 2].axis("off")

    # Plot curves
    for row_idx, (block_label, _, block_color) in enumerate(BLOCK_ROWS):
        responses_block = RESPONSES_BY_BLOCK[block_label]
        display_block_label = block_label.replace("LC4 & LPLC2", "LC4 &\nLPLC2")
        for col_idx, (side, ms) in enumerate(PANEL_ORDER):
            ax = axes[row_idx, col_map[col_idx]]
            style_axis_keep_xline(ax)
            ax.set_ylim(-0.3, 1.05)
            neurons_ctrl = get_target_neurons(responses_ctrl, TARGET_TYPE, side)
            neurons_block = get_target_neurons(responses_block, TARGET_TYPE, side)
            # =====================================================
            # 1. Select the peak coordinate ONLY from CTRL.
            # =====================================================
            resp_ctrl, ctrl_peak, ctrl_peak_coord, ctrl_peak_nid = get_ctrl_peak_reference(
                responses_ctrl,
                ms,
                neurons_ctrl
            )

            if resp_ctrl is None:
                continue

            # =====================================================
            # 2. For blocked conditions, use the SAME coordinate.
            #    Do not search for a new peak in the blocked data.
            # =====================================================
            resp_block = get_trace_at_ctrl_peak_coord(
                responses_block,
                ms,
                neurons_block,
                ctrl_peak_coord,
                preferred_nid=ctrl_peak_nid
            )

            # =====================================================
            # 3. Normalization rule.
            #    - CTRL: normalized by its own min/max.
            #    - LC4 / LPLC2: blocked curve normalized by its own min/max.
            #    - LC4 & LPLC2: blocked curve is NOT normalized.
            # =====================================================
            resp_ctrl_plot, resp_block_plot = prepare_plot_traces(
                resp_ctrl,
                resp_block,
                block_label
            )

            x = np.arange(len(resp_ctrl_plot))
            ax.fill_between(
                x,
                0,
                resp_ctrl_plot,
                color=COLOR_CTRL,
                alpha=0.1,
                linewidth=0
            )
            ax.plot(
                x,
                resp_ctrl_plot,
                color=COLOR_CTRL,
                linewidth=3.2,
                solid_capstyle="round",
                solid_joinstyle="round"
            )

            if resp_block_plot is not None:
                resp_block_plot = match_length(resp_block_plot, len(resp_ctrl_plot))
                ax.plot(
                    x,
                    resp_block_plot,
                    color=block_color,
                    linewidth=3.6,
                    solid_capstyle="round",
                    solid_joinstyle="round"
                )

            tau_val = float(ms.replace("ms", "")) / 1000.0
            rv_curve = generate_rv_curve(tau_val)
            rv_curve = match_length(rv_curve, len(resp_ctrl_plot))
            if np.max(rv_curve) > 0:
                rv_curve = rv_curve / np.max(rv_curve)
            ax.plot(
                x,
                rv_curve,
                color=COLOR_RV,
                linestyle="--",
                linewidth=2.4,
                alpha=0.85,
                solid_capstyle="round"
            )
            ax.set_xlim(0, len(resp_ctrl_plot) - 1)
            if col_idx==0:
                ax.text(-0.10,0.50,display_block_label,transform=ax.transAxes,rotation=90,ha="right",va="center",fontsize=36,color="black")

    # =========================
    # Shared legend
    # =========================
    handles = [
        Line2D([0],[0],color=COLOR_CTRL,lw=3.2,label="Intact"),
        Line2D([0],[0],color=COLOR_BLOCK,lw=3.2,label="Blocked"),
        Line2D([0],[0],color=COLOR_RV,lw=3.2,ls="--",label="r/v")
    ]
    fig.legend(handles=handles, loc="center left", bbox_to_anchor=(0.90,0.50), frameon=False, fontsize=36, handlelength=1.8, borderpad=0.2, labelspacing=0.5)

    out_png = os.path.join(OUT_DIR,"e.png")
    fig.savefig(out_png,dpi=600,bbox_inches="tight",pad_inches=0.03)
    plt.close(fig)
    gc.collect()
    print(f"Saved: {out_png}")

plot_dnp01_block_change_3x4_paper_style()

100%|██████████| 3162/3162 [00:17<00:00, 183.62it/s]


Saved: ../results/Figure4/e.png
